# Multi-session dimensionality (50-unit populations)

Companion to notebook 06. Three differences from it, all deliberate:

1. **Populations are equalised to 50 neurons**, not 25, so only recordings with
   at least 50 sorted units enter. This costs populations but gives a
   50-component spectrum, which is what the maximum-likelihood fit needs.
2. **The exponent is estimated by maximum likelihood** (Clauset, Shalizi &
   Newman 2009, via the `powerlaw` package) rather than by least squares on
   log-log axes. It is reported converted to rank-exponent units,
   `1/(alpha_hat - 1)`, so it is comparable with the single-session chapters.
3. **Two direct dimensionality measures are computed alongside it** --- the
   participation ratio `D_PR = (sum lambda)^2 / sum lambda^2` and `D_90`, the
   number of components needed for 90 % of the variance. Neither requires a
   fitted range, so they test whether the exponent is a good proxy for
   dimensionality at all.

Everything is read from `../data/dandi_binned_1s_rat.npz`; the notebook needs
nothing from the other notebooks and runs end to end in a few minutes.


In [ ]:
import os, sys, hashlib, warnings
import numpy as np, pandas as pd, powerlaw
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon

warnings.filterwarnings("ignore")                     # powerlaw is chatty about xmin
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(HERE, "..", "scripts")))
from dandi_data import load_pack

STATES  = ("Wake", "NREM", "REM")
COLORS  = {"Wake": "#D55E00", "NREM": "#0072B2", "REM": "#009E73"}
N_SUB   = 50          # neurons every population is cut to
DRAWS   = 50          # subsample repeats per population and state
MIN_BINS = 300        # a state needs 5 minutes to give a spectrum
SEED    = 1
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "legend.fontsize": 7.5, "savefig.dpi": 300,
                     "figure.constrained_layout.use": True})


def pop_rng(tag):
    """Deterministic per-population stream, so a population's value does not
    depend on the order in which populations are processed."""
    h = int.from_bytes(hashlib.sha256(("pr50|" + tag).encode()).digest()[:4], "big")
    return np.random.default_rng(SEED ^ h)


def spectrum(M):
    """Eigenvalues of the neuron covariance of a (bins x neurons) matrix.

    Neurons are mean-centred but NOT z-scored: the spectrum is of the
    covariance, so a neuron contributes in proportion to its own variance.
    """
    M = M - M.mean(0)
    ev = np.linalg.svd(M, compute_uv=False) ** 2
    return ev / ev.sum()


## Populations

A *population* is the neurons of one brain region in one recording. Eleven
recordings have electrodes in two regions and therefore contribute two
populations, which are kept apart: pooling two circuits into one covariance
matrix would mix two different things.


In [ ]:
pops = [p for p in load_pack(min_units=N_SUB)
        if all((p["state"] == s).sum() >= MIN_BINS for s in STATES)]
inv = pd.DataFrame([dict(tag=p["tag"].split("_ses-")[-1], group=p["group"],
                         n_units=p["X"].shape[1],
                         **{f"bins_{s}": int((p["state"] == s).sum()) for s in STATES})
                    for p in pops])
print(f"{len(pops)} populations from {inv.tag.nunique()} recordings")
print(inv.group.value_counts().to_string())


## Measures

For each population and state, `DRAWS` random subsets of `N_SUB` neurons are
drawn and four quantities averaged over them:

| quantity | definition | needs a fitted range? |
|---|---|---|
| `D_pr` | participation ratio, (Σλ)² / Σλ² | no |
| `D_90` | components reaching 90 % of the variance | no |
| `alpha_mle` | Pareto tail index above the KS-selected cutoff | the cutoff is chosen from the data |
| `alpha_conv` | the same fit as a rank exponent, 1/(α̂ − 1) | — |

The active fraction `s` is computed on the **fixed** population, not on the
subsample: sparseness is a property of the recorded population, while only the
spectrum needs equalising.


In [ ]:
rows, skipped = [], []
for p in pops:
    rng = pop_rng(f"{p['group']}|{p['tag']}")
    rec = dict(tag=p["tag"].split("_ses-")[-1], group=p["group"], n_units=p["X"].shape[1])
    for s in STATES:
        M = p["X"][p["state"] == s]
        live = np.flatnonzero(M.sum(0) >= 10)          # a unit silent in this state cannot be drawn
        if len(live) < N_SUB:
            # Having >= 50 sorted units is not enough: a population also needs
            # 50 units that are ACTIVE in every state, or it cannot be equalised.
            skipped.append((p["tag"].split("_ses-")[-1], p["group"], s, len(live)))
            break
        pr_, d90_, al_, dks_, nt_ = [], [], [], [], []
        for _ in range(DRAWS):
            ev = np.sort(spectrum(M[:, rng.choice(live, N_SUB, replace=False)]))[::-1]
            pr_.append((ev.sum() ** 2) / (ev ** 2).sum())
            d90_.append(int(np.searchsorted(np.cumsum(ev), 0.90) + 1))
            f = powerlaw.Fit(ev, discrete=False, verbose=False).power_law
            al_.append(f.alpha); dks_.append(f.D); nt_.append(f.n)
        rec[f"D_pr_{s}"]       = float(np.mean(pr_))
        rec[f"D_90_{s}"]       = float(np.mean(d90_))
        rec[f"alpha_mle_{s}"]  = float(np.mean(al_))
        rec[f"alpha_conv_{s}"] = float(np.mean(1.0 / (np.array(al_) - 1.0)))
        rec[f"D_ks_{s}"]       = float(np.mean(dks_))
        rec[f"n_tail_{s}"]     = float(np.mean(nt_))
        rec[f"s_{s}"]          = float((M > 0).mean())
    else:
        rows.append(rec)

P50 = pd.DataFrame(rows)
print(f"{len(P50)} populations usable; {len(skipped)} dropped for too few active units:")
for t in skipped:
    print(f"   {t[0]:22s} {t[1]:20s} {t[2]:5s} only {t[3]} active units")
P50.round(4).to_csv("multisession_50units_clauset.csv", index=False)
print(P50[[f"{q}_{s}" for q in ("D_pr", "D_90", "alpha_conv", "D_ks", "s")
           for s in STATES]].median().round(3).to_string())


## Which state is lower-dimensional?

Paired across populations, so each recording is compared with itself.


In [ ]:
tests = []
for q in ("D_pr", "D_90", "alpha_conv", "s"):
    for a, b in (("Wake", "NREM"), ("Wake", "REM"), ("NREM", "REM")):
        d = P50[f"{q}_{a}"] - P50[f"{q}_{b}"]
        tests.append(dict(quantity=q, comparison=f"{a} > {b}",
                          n_greater=int((d > 0).sum()), n=len(d),
                          median_diff=round(float(d.median()), 3),
                          p_wilcoxon=float(wilcoxon(d).pvalue)))
T = pd.DataFrame(tests)
T.to_csv("multisession_50units_tests.csv", index=False)
print(T.to_string(index=False))


## Exponent by state, maximum likelihood

The multi-session exponent, estimated the Clauset way and converted to rank
units. Open circles are the individual populations; the filled marker is the
median with the interquartile range.


In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.5))
for i, s in enumerate(STATES):
    v = P50[f"alpha_conv_{s}"].values
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    ax.plot(np.full(len(v), i) + np.random.default_rng(i).uniform(-.11, .11, len(v)),
            v, "o", ms=2.8, mfc="none", mew=.6, color=COLORS[s], alpha=.55, zorder=1)
    ax.errorbar(i + 0.27, med, yerr=[[med - q1], [q3 - med]], fmt="o", ms=7.5,
                color=COLORS[s], elinewidth=1.6, capsize=4, zorder=3)
    ax.annotate(f"{med:.2f}", (i + 0.27, med), textcoords="offset points",
                xytext=(11, -3), fontsize=8, color=COLORS[s])
ax.set_xticks(range(3)); ax.set_xticklabels(STATES); ax.set_xlim(-0.45, 2.75)
ax.set_ylabel(r"exponent  $1/(\hat{\alpha}-1)$   (50 units, max. likelihood)")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(f"{FIG_DIR}/fig4_exponent_mle_50units.png")
plt.show()


## Is the exponent a good measure of dimensionality?

Three scatters, one marker per population and state. The first compares the two
direct measures with each other; the second asks how well the exponent tracks
them; the third is the sparseness-versus-dimensionality relation the thesis is
built around.


In [ ]:
long = pd.concat([P50.assign(state=s, D_pr=P50[f"D_pr_{s}"], D_90=P50[f"D_90_{s}"],
                             a_conv=P50[f"alpha_conv_{s}"], s_act=P50[f"s_{s}"])
                  [["tag", "group", "state", "D_pr", "D_90", "a_conv", "s_act"]]
                  for s in STATES])

panels = [("fig5a_Dpr_vs_D90.png", "D_pr", "D_90",
           r"participation ratio  $D_{\mathrm{PR}}$",
           r"components for 90 % variance  $D_{90}$"),
          ("fig5b_alpha_vs_Dpr.png", "a_conv", "D_pr",
           r"tail index converted,  $1/(\hat{\alpha}-1)$",
           r"participation ratio  $D_{\mathrm{PR}}$"),
          ("fig5c_sparseness_vs_Dpr.png", "s_act", "D_pr",
           r"mean active fraction  $s$",
           r"participation ratio  $D_{\mathrm{PR}}$")]

corr = []
for fname, xc, yc, xl, yl in panels:
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    for s in STATES:
        d = long[long.state == s]
        r_ = spearmanr(d[xc], d[yc])
        corr.append(dict(panel=fname, state=s, rho=round(r_.statistic, 3),
                         p=float(r_.pvalue), n=len(d)))
        ax.plot(d[xc], d[yc], "o", ms=5, mfc="none", mew=1.2, color=COLORS[s],
                label=f"{s}   $\\rho$ = {r_.statistic:+.2f}"
                      + ("" if r_.pvalue < .05 else " (n.s.)"))
    rp = spearmanr(long[xc], long[yc])
    corr.append(dict(panel=fname, state="pooled", rho=round(rp.statistic, 3),
                     p=float(rp.pvalue), n=len(long)))
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(0.06)
    fig.savefig(f"{FIG_DIR}/{fname}")
    plt.show()
C = pd.DataFrame(corr)
C.to_csv("multisession_50units_correlations.csv", index=False)
print(C.to_string(index=False))


## Firing-rate distribution and the exclusion threshold

Every unit's session-wide rate, pooled over all recordings in the pack, with the
4.71 Hz threshold marked. In Achilles the threshold falls in an empty gap
between two groups of neurons; pooled across recordings the distribution is
unimodal and roughly log-normal, so the threshold is a cut in a continuous
distribution rather than a boundary the data draw.


In [ ]:
z = np.load(os.path.join(HERE, "..", "data", "dandi_binned_1s_rat.npz"), allow_pickle=True)
rate_all = np.concatenate([z[k] for k in z.files if k.startswith("rate__")])
THRESH = 4.71

fig, ax = plt.subplots(figsize=(4.4, 3.2))
bins = np.geomspace(max(rate_all.min(), 1e-3), rate_all.max() * 1.05, 45)
ax.hist(rate_all, bins=bins, color="0.55", edgecolor="white", lw=.4)
ax.axvline(THRESH, color=COLORS["Wake"], lw=1.4, ls="--")
ax.annotate(f"{THRESH} Hz\n({int((rate_all > THRESH).sum())} excluded)",
            (THRESH, ax.get_ylim()[1]), textcoords="offset points",
            xytext=(6, -22), fontsize=7.5, color=COLORS["Wake"])
ax.set_xscale("log")
ax.set_xlabel("session-wide firing rate (Hz)")
ax.set_ylabel("neurons")
ax.set_title(f"{len([k for k in z.files if k.startswith('rate__')])} recordings, "
             f"{len(rate_all):,} units", loc="left", fontsize=8.5)
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(f"{FIG_DIR}/fig4_rate_hist_all.png")
plt.show()
print(f"median rate {np.median(rate_all):.3f} Hz | above threshold "
      f"{100 * np.mean(rate_all > THRESH):.1f}%")


## Outputs

| file | contents |
|---|---|
| `multisession_50units_clauset.csv` | per population and state: `D_pr`, `D_90`, `alpha_mle`, `alpha_conv`, `D_ks`, `n_tail`, `s` |
| `multisession_50units_tests.csv` | the paired state comparisons |
| `multisession_50units_correlations.csv` | the per-state and pooled correlations behind each scatter |
| `figures/fig4_exponent_mle_50units.png` | exponent by state |
| `figures/fig5a_Dpr_vs_D90.png`, `fig5b_alpha_vs_Dpr.png`, `fig5c_sparseness_vs_Dpr.png` | the three scatters |
| `figures/fig4_rate_hist_all.png` | rate distribution and threshold |

Panel letters are not drawn in the figures; they are assigned in the LaTeX float
so the files and the manuscript cannot disagree.
